In [ ]:
import polars as pl
import polars.selectors as cs

In [ ]:
from hexagonal.files.spec import get_polars_dataframe

In [ ]:
from splink import block_on, DuckDBAPI, Linker, SettingsCreator
import splink.exploratory as ex
from splink.blocking_analysis import count_comparisons_from_blocking_rule, cumulative_comparisons_to_be_scored_from_blocking_rules_chart, n_largest_blocks
import splink.comparison_level_library as cll
import splink.comparison_library as cl

In [ ]:
def normaliser(colonne):
    if isinstance(colonne, str):
        colonne = pl.col(colonne)
    return (
        colonne
        .str.normalize("NFKD")  # splitter les diacritiques
        .str.to_lowercase()
        .str.replace_all(r"œ", "oe")    # à traiter spécifiquement en français
        .str.replace_all(r"\p{Nonspacing Mark}", "")  # retirer les diacritiques
        .str.replace_all(r"\pP", " ")  # ponctuation
        .str.replace_all(r"\s\s+", " ")
        .str.strip_chars()
    )

In [ ]:
code_departement = pl.when(
    pl.col("code_commune").str.slice(0, 2).is_in(["97", "98"])
).then(
    pl.col("code_commune").str.slice(0, 3)
).otherwise(
    pl.col("code_commune").str.slice(0, 2)
).alias("code_departement")

In [ ]:
nom_complet = pl.format("{} {}", normaliser("prenom"), normaliser("nom")).alias("nom_complet")
noms = [
    normaliser("prenom"),
    normaliser("nom"),
    nom_complet
]

In [ ]:
maires = get_polars_dataframe("data/02_clean/rne/conseillers_municipaux.csv").filter(
    pl.col("fonction").is_in(["Maire", "Maire délégué"])
).select(
    "code_commune",
    *noms,
    "sexe",
    pl.col("fonction").str.to_lowercase(),
    code_departement,
).unique(
    ["code_commune", "nom", "prenom", "sexe"], keep="first"
).sort(
    ["code_commune", "nom", "prenom"]
).with_row_index().with_columns(
    pl.format("MAI{}", pl.col("index"))
)

In [ ]:
parrainages = get_polars_dataframe("data/03_main/elections/2022-presidentielle-parrainages.csv").with_row_index().with_columns(
    index="CC" + pl.col("index").cast(pl.String()).str.zfill(5)
)

In [ ]:
parrainages_maires = parrainages.filter(
    pl.col("mandat").is_in(["maire", "maire délégué"])
).select(
    "index",
    pl.col("code_circonscription").alias("code_commune"),
    *noms,
    "sexe",
    "candidat",
    pl.col("mandat").alias("fonction"),
).with_columns(code_departement)

In [ ]:
api = DuckDBAPI()

api.register_multiple_tables(
    [maires, parrainages_maires],
    ["maires", "parrainages"]
)

In [ ]:
count_comparisons_from_blocking_rule(
    table_or_tables=["maires", "parrainages"],
    blocking_rule=block_on("code_departement"),
    link_type="link_only",
    db_api=api,
    unique_id_column_name="index",
)

In [ ]:
cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=["maires", "parrainages"],
    blocking_rules=[
        block_on("prenom"),
        block_on("nom"),
        block_on("code_commune"),
        block_on("code_departement"),
    ],
    link_type="link_only",
    db_api=api,
    unique_id_column_name="index",
)

In [ ]:
full_name_comparison = cl.CustomComparison(
    output_column_name="Nom complet",
    comparison_levels=[
        cll.CustomLevel(
            sql_condition="(prenom_l IS NULL OR prenom_r IS NULL) and (nom_l IS NULL OR nom_r IS NULL)",
            label_for_charts="Null",
        ).configure(is_null_level=True),
        cll.ExactMatchLevel("nom_complet", term_frequency_adjustments=True),
        cll.JaroWinklerLevel("nom_complet", 0.9),
        cll.ColumnsReversedLevel("nom", "prenom"),
        {
            "sql_condition": "jaro_winkler_similarity(prenom_l, nom_r) + jaro_winkler_similarity(nom_l, prenom_r) >= 1.8",
            "label_for_charts": "switched + jaro_winkler_similarity >= 1.8",
        },
        {
            "sql_condition": """
                (
                    contains(prenom_l, prenom_r) or contains(prenom_r, prenom_l)
                ) and (
                    contains(nom_l, nom_r) or contains(nom_r, nom_l)
                )""",
            "label_for_charts": "contained"
        },
        cll.ElseLevel(),
    ]
)

code_commune_comparison = {
    "output_column_name": "Commune",
    "comparison_levels": [
        {
            "sql_condition": "code_commune_l IS NULL AND code_commune_r IS NULL",
            "label_for_charts": "Null",
            "is_null_level": True,
        },
        cll.ExactMatchLevel("code_commune"),
        cll.ExactMatchLevel("code_departement", term_frequency_adjustments=True),
        cll.ElseLevel(),
    ]
}

In [ ]:
blocks = [
    block_on("prenom"),
    block_on("nom"),
    block_on("code_commune"),
    block_on("code_departement")
]

comparisons = [
    full_name_comparison,
    code_commune_comparison,
    cl.ExactMatch("sexe").configure(term_frequency_adjustments=True),
]

settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="index",
    blocking_rules_to_generate_predictions=blocks,
    comparisons=comparisons,
    retain_intermediate_calculation_columns=True,
)

linker = Linker(
    ["maires", "parrainages"],
    settings,
    api
)

In [ ]:
deterministic_rules = [
    block_on("nom", "prenom", "sexe"),
]

linker.training.estimate_probability_two_random_records_match(
    deterministic_rules,
    recall=0.90
)

In [ ]:
linker.training.estimate_u_using_random_sampling(1e7)

In [ ]:
session_departement = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("code_departement"), estimate_without_term_frequencies=True
)

In [ ]:
session_departement.m_u_values_interactive_history_chart()

In [ ]:
session_prenom = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("prenom"), estimate_without_term_frequencies=True
)

In [ ]:
session_prenom.m_u_values_interactive_history_chart()

In [ ]:
session_departement.match_weights_interactive_history_chart()

In [ ]:
linker.visualisations.m_u_parameters_chart()

In [ ]:
predictions = linker.inference.predict(threshold_match_probability=0.2)

In [ ]:
linker.visualisations.parameter_estimate_comparisons_chart()

In [ ]:
linker.visualisations.waterfall_chart(predictions.as_record_dict(limit=10))

In [ ]:
linker.visualisations.comparison_viewer_dashboard(predictions, "test.html", overwrite=True)